# 03. Clasificarea unei imagini proprii cu DoReFaNet

Încărcați o imagine JPG, PNG sau BMP în directorul **images** asociat notebook-urilor QNN. Executați celulele în ordine cu **Shift + Enter**. Implicit este aleasă cea mai recentă imagine; puteți introduce un nume de fișier în secțiunea 3.

Modelul prezice una dintre cele 1.000 de clase ImageNet pentru **întreaga imagine**; nu localizează obiecte multiple. [Laboratorul 02](02_dorefanet-imagenet-samples.ipynb) folosește, în schimb, exemple cu etichetă cunoscută.

![Topologia DoReFaNet](../dorefanet-topology.svg)

Prima convoluție și straturile complet conectate rulează pe procesorul ARM, iar straturile intermediare cuantizate pe FPGA. Închideți celelalte kerneluri care folosesc FPGA. Mențineți activ swap-ul, dacă este necesar pe placa de 512 MB. Utilizați pachetul instalat cu importurile `Xlnk` nefolosite deja eliminate; nu este necesară reinstalarea.

*Traducerea păstrează fluxul de inferență al versiunii engleze; execuția hardware a acestei copii nu a fost validată separat.*

## 1. Identificarea parametrilor instalați și a imaginilor

Executați această celulă. Ea verifică fișierele înainte de inițializarea FPGA. `NOTEBOOK_DIR` pornește de la directorul de lucru, apoi verifică directorul părinte și locația standard a plăcii. Încărcați imaginile în subdirectorul său **images** și reexecutați celula.

In [ ]:
import os
import gc
import pickle
import random
import time
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image
from IPython.display import display

import qnn
from qnn import Dorefanet, utils

QNN_ROOT = Path(qnn.__file__).resolve().parent
QNN_PARAM_DIR = QNN_ROOT / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "imagenet-classes.pkl").is_file():
    if (NOTEBOOK_DIR.parent / "imagenet-classes.pkl").is_file():
        NOTEBOOK_DIR = NOTEBOOK_DIR.parent
    else:
        NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
SAMPLE_DIR = NOTEBOOK_DIR / "images"
LABEL_FILE = NOTEBOOK_DIR / "imagenet-classes.pkl"

required_files = [QNN_PARAM_DIR / name for name in (
    "dorefanet-layers.json", "dorefanet-conv0.npy",
    "dorefanet-fc-normalized.npy", "ilsvrc_pp_mean.npy")]
required_files.append(LABEL_FILE)
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing:\n" + "\n".join(missing))
if not SAMPLE_DIR.is_dir():
    raise FileNotFoundError("Create/upload images into: " + str(SAMPLE_DIR))
image_files = sorted(path for path in SAMPLE_DIR.iterdir()
                     if path.is_file() and path.suffix.lower() in
                     (".jpg", ".jpeg", ".png", ".bmp"))
if not image_files:
    raise FileNotFoundError("Upload a JPG/PNG/BMP image into " + str(SAMPLE_DIR))
print("Python package:", QNN_ROOT)
print("Notebook data:", NOTEBOOK_DIR)
print("Available images:", len(image_files))

## 2. Încărcarea ponderilor și etichetelor, apoi inițializarea modelului

Executați celula următoare și așteptați mesajul **Model ready**. Sunt încărcate dicționarele de ponderi NumPy existente și cele 1.000 de etichete ImageNet, după care este inițializat acceleratorul FPGA. Se folosește configurația implicită a rețelei, care identifică deja locația pachetului instalat.

Un mesaj este afișat înaintea fiecărei încărcări mari. Dacă această celulă eșuează, nu continuați cu inferența. Reexecutarea eliberează clasificatorul precedent înaintea creării unuia nou. Fișierele pickle și NumPy provin din pachetul de încredere folosit de laborator.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
# Avoid holding old weight dictionaries while reloading this cell.
for variable in ("conv0_weights", "fc_weights"):
    globals().pop(variable, None)
gc.collect()

print("Loading first-convolution weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-conv0.npy"),
                       encoding="latin1", allow_pickle=True).item()
print("Loading fully connected weights; please wait...", flush=True)
fc_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-fc-normalized.npy"),
                    encoding="latin1", allow_pickle=True).item()
for key in ("conv0/W", "conv0/T"):
    if key not in conv0_weights:
        raise KeyError("Missing weight: " + key)
for key in ("fc0/Wn", "fc0/bn", "fc1/Wn", "fc1/bn", "fct/W"):
    if key not in fc_weights:
        raise KeyError("Missing weight: " + key)

with LABEL_FILE.open("rb") as label_stream:
    classes = pickle.load(label_stream, encoding="latin1")
if set(classes.keys()) != set(range(1000)):
    raise ValueError("Expected ImageNet labels indexed from 0 to 999.")
names = {index: classes[index][1].split(",")[0] for index in classes}
synset_to_index = {classes[index][0]: index for index in classes}

print("Initializing the FPGA...", flush=True)
classifier = Dorefanet()
try:
    net = classifier.load_network()
    classifier.init_accelerator()
except BaseException:
    classifier.deinit_accelerator()
    raise
print("Model ready. ImageNet classes:", len(classes))

## 3. Alegerea și afișarea imaginii

Păstrați `IMAGE_NAME = None` pentru selectarea celei mai recente imagini din **images**. Pentru un anumit fișier, înlocuiți `None` cu numele între ghilimele. Executați celula și apoi secțiunile **4–8**. Imaginile proprii nu au o etichetă de referință verificată; rezultatele sunt predicții, nu măsurători ale acurateței.

In [ ]:
IMAGE_NAME = None  # Example: replace None with an existing filename string.
image_path = max(image_files, key=lambda path: path.stat().st_mtime) if IMAGE_NAME is None else SAMPLE_DIR / IMAGE_NAME
if not image_path.is_file():
    raise FileNotFoundError("Image not found: " + str(image_path))
img, img_class = classifier.load_image(str(image_path))
expected_index = None  # No ground-truth label for a user image.
print("Image:", image_path.name)
print("Correct class:", names[expected_index] if expected_index is not None
      else "not available from this filename")
with Image.open(str(image_path)) as opened:
    preview = opened.convert("RGB")
preview.thumbnail((480, 360))
display(preview)
stage_seconds = {}

## 4. Prima convoluție pe procesorul ARM

Executați celula următoare. Stratul folosește ponderile originale în precizie completă. Ieșirea sa este pragată pentru reprezentarea cerută de straturile cuantizate FPGA. Timpul afișat include convoluția și pragarea.

In [ ]:
start = time.perf_counter()
conv0 = utils.conv_layer(img, conv0_weights["conv0/W"], stride=4)
conv0 = utils.threshold(conv0, conv0_weights["conv0/T"])
stage_seconds["First convolution + threshold (CPU)"] = time.perf_counter() - start
print("CPU first stage: {:.3f} s".format(
    stage_seconds["First convolution + threshold (CPU)"]))

## 5. Straturile cuantizate pe FPGA

Executați celula următoare. Același accelerator execută secvența configurată de straturi cuantizate. Pregătirea bufferelor și conversia ieșirii sunt în afara timpului măsurat pentru apelul de inferență.

Această temporizare descrie **apelul de offload**, cu costul runtime inclus, nu numai ciclurile aritmetice ale FPGA-ului.

In [ ]:
out_dim = net["merge4"]["output_dim"]
out_ch = net["merge4"]["output_channels"]
conv_output = classifier.get_accel_buffer(out_ch, out_dim)
conv_input = np.ascontiguousarray(classifier.prepare_buffer(conv0))
start = time.perf_counter()
classifier.inference(conv_input, conv_output)
stage_seconds["Quantized-layer offload (FPGA)"] = time.perf_counter() - start
conv_output_hw_raw = conv_output.copy()
conv_output = classifier.postprocess_buffer(conv_output)
if not np.all(np.isfinite(conv_output)):
    raise ValueError("Non-finite accelerator output. Check model initialization.")
print("FPGA offload call: {:.3f} s".format(
    stage_seconds["Quantized-layer offload (FPGA)"]))

## 6. Straturile complet conectate pe procesorul ARM

Executați celula următoare. Sunt păstrate normalizarea și calculele straturilor complet conectate din rețeaua originală. O ieșire FPGA formată integral din zerouri este raportată explicit, evitând împărțirea la zero.

In [ ]:
scale = np.max(conv_output)
if scale <= 0:
    raise RuntimeError("Accelerator output is all zero. Stop and check initialization before continuing.")
fc_input = conv_output / scale
if fc_input.size != fc_weights["fc0/Wn"].shape[0]:
    raise ValueError("Accelerator output size does not match the fully connected weights.")
start = time.perf_counter()
fc0_out = utils.fully_connected(fc_input, fc_weights["fc0/Wn"], fc_weights["fc0/bn"])
fc0_out = utils.quantize(utils.qrelu(fc0_out), 2)
fc1_out = utils.fully_connected(fc0_out, fc_weights["fc1/Wn"], fc_weights["fc1/bn"])
fc1_out = utils.qrelu(fc1_out)
fct_b = np.zeros((fc_weights["fct/W"].shape[1],))
fct_out = utils.fully_connected(fc1_out, fc_weights["fct/W"], fct_b)
stage_seconds["Fully connected layers (CPU)"] = time.perf_counter() - start
print("CPU fully connected stages: {:.3f} s".format(
    stage_seconds["Fully connected layers (CPU)"]))

## 7. Cele cinci predicții principale

Barele prezintă cele mai mari cinci scoruri softmax ImageNet pentru imagine, obținute prin fluxul combinat CPU/FPGA. Aceste scoruri nu reprezintă măsurarea acurateței și nu provin dintr-un alt model rulat exclusiv în software. Un obiect care nu face parte din clasele de antrenare va primi totuși o predicție.

In [ ]:
if np.asarray(fct_out).shape != (len(classes),) or not np.all(np.isfinite(fct_out)):
    raise ValueError("Expected 1,000 finite classification scores.")
out = utils.softmax(fct_out)
topn = utils.get_topn_indexes(out, 5)
print("Correct class:", names[expected_index] if expected_index is not None else "unknown")
for index in topn:
    print("{:25s} {:7.2%}".format(names[int(index)], float(out[index])))
if expected_index is not None:
    print("Top-1:", "correct" if int(topn[0]) == expected_index else "wrong")
    print("Top-5:", "correct class included" if expected_index in topn
          else "correct class not included")
fig, ax = plt.subplots(figsize=(8, 3.5))
positions = np.arange(len(topn))
colors = ["green" if int(index) == expected_index else "steelblue" for index in topn]
ax.barh(positions, out[topn], color=colors)
ax.set_yticks(positions)
ax.set_yticklabels([names[int(index)] for index in topn])
ax.invert_yaxis()
ax.set_xlim(0, min(1.0, float(np.max(out[topn])) * 1.25))
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Softmax score (axis scaled to the displayed predictions)")
ax.set_title("Top-5 predictions for your image")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)

## 8. Compararea timpilor măsurați pe etape

Executați celula următoare. Sunt afișate cele trei etape cronometrate și suma lor. Etapele CPU și FPGA execută **straturi diferite**; aceasta este împărțirea sarcinii, nu o măsurare a accelerării aceleiași operații de către FPGA față de CPU.

Suma exclude citirea imaginii, pregătirea și conversia bufferelor, softmax, graficele și pauzele dintre celule. Nu trebuie interpretată ca FPS al camerei sau debit al aplicației complete. Timpurile nu sunt salvate într-un fișier comun cu alte notebook-uri.

In [ ]:
stage_order = ["First convolution + threshold (CPU)",
               "Quantized-layer offload (FPGA)",
               "Fully connected layers (CPU)"]
if any(name not in stage_seconds for name in stage_order):
    raise RuntimeError("Run all three computation stages for the current image first.")
values = [stage_seconds[name] for name in stage_order]
for name, seconds in zip(stage_order, values):
    print("{}: {:.3f} s".format(name, seconds))
print("Sum of measured stages: {:.3f} s".format(sum(values)))
fig, ax = plt.subplots(figsize=(9, 3.5))
positions = np.arange(len(stage_order))
ax.barh(positions, values, color=["steelblue", "darkorange", "steelblue"])
ax.set_yticks(positions)
ax.set_yticklabels(stage_order)
ax.invert_yaxis()
ax.set_xlabel("Measured stage time (seconds)")
ax.set_title("CPU and FPGA stages of the same inference")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Opțional: compararea acelorași straturi cuantizate în software

Păstrați `RUN_SOFTWARE_COMPARISON = False` pentru a omite testul mai lent. Setați `True` pentru a executa **aceeași intrare a straturilor cuantizate** cu backend-ul C++ software instalat, fără a descărca alt bitstream. Se compară ieșirile brute și duratele celor două apeluri de inferență. Prima convoluție și straturile complet conectate sunt excluse din ambele durate; nu este o a doua clasificare cap-coadă.

Executați această secțiune înaintea eliberării resurselor. Backend-ul software este eliberat inclusiv în cazul unei erori.

In [ ]:
RUN_SOFTWARE_COMPARISON = False
if RUN_SOFTWARE_COMPARISON:
    classifier_sw = Dorefanet("python_sw")
    try:
        classifier_sw.load_network()
        classifier_sw.init_accelerator(bit_file=None)
        sw_output = classifier_sw.get_accel_buffer(out_ch, out_dim)
        start = time.perf_counter()
        classifier_sw.inference(conv_input, sw_output)
        sw_seconds = time.perf_counter() - start
        hw_seconds = stage_seconds["Quantized-layer offload (FPGA)"]
        print("Same quantized layers — FPGA call: {:.3f} s; software call: {:.3f} s".format(
            hw_seconds, sw_seconds))
        print("Raw outputs exactly equal:", np.array_equal(conv_output_hw_raw, sw_output))
        if hw_seconds > 0:
            print("Software / FPGA call time: {:.2f}x".format(sw_seconds / hw_seconds))
    finally:
        classifier_sw.deinit_accelerator()
        del classifier_sw
        globals().pop("sw_output", None)
else:
    print("Software comparison skipped.")

## 10. Finalizare și eliberarea resurselor

După terminarea testelor, executați celula următoare. Este apelată metoda de curățare a modelului și sunt eliminate tablourile intermediare și dicționarele de ponderi voluminoase. Nu se folosește `Xlnk` și nu se face resetarea globală a dispozitivului.

Pentru o nouă execuție după curățare, reluați secțiunea 2. Opriți kernelul înainte de a deschide un alt notebook care utilizează FPGA.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
for variable in ("conv0_weights", "fc_weights", "img", "conv0", "conv_input",
                 "conv_output", "conv_output_hw_raw", "fc_input", "fc0_out", "fc1_out", "fct_out"):
    globals().pop(variable, None)
gc.collect()
print("Classifier released and large Python arrays cleared.")

**Validare:** fluxul Python și tratarea erorilor au fost verificate off-board cu interfețe native/accelerator simulate și funcțiile NumPy originale. Execuția FPGA, comportamentul bibliotecii Darknet native și calitatea predicțiilor trebuie încă verificate pe placa PYNQ.